In [ ]:
import torch
from ultralytics import YOLO
from spdconv import register_spdconv

# SART: YAML'daki "SPDConv" adini BU klasordeki spdconv.py'ye baglar.
# Cagrilmazsa ultralytics'e elle eklenmis inn_modules_v2.SPDConv devreye girer;
# o surumun state_dict anahtarlari farkli (conv.conv.weight vs conv.weight),
# yani SPD-Conv checkpoint'i SESSIZCE transfer olmaz (hata vermez, sadece
# transfer sayisi duser).
register_spdconv()

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu" 
print(f"Kullanilan cihaz: {'GPU (cuda:0)' if device == 0 else 'CPU'}")
if device == "cpu":
        print("UYARI: GPU bulunamadi, egitim CPU'da yapilacak ve cok yavas olabilir.")

# --- Veri seti yolu ---
# Once depo icindeki dataset/ klasoruna bakar (README'deki kurulum),
# bulamazsa gelistirme makinesindeki konuma duser.
import os

_CANDIDATES = [
    "../dataset/data.yaml",
    "../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
]
DATA = next((p for p in _CANDIDATES if os.path.exists(p)), _CANDIDATES[0])
print("veri seti:", os.path.abspath(DATA), "->", os.path.exists(DATA))

In [ ]:
# --- SPD-Conv + P2 ablasyonu ---
# Baseline'a gore IKI degisiklik (bu ablasyonun tanimi):
#   1) butun stride-2 downsample'lar SPDConv (space-to-depth, bilgi kaybi yok)
#   2) P2/4 tespit dali eklendi -> Detect 4 seviyeli [P2,P3,P4,P5]
#
# P2 NEDEN: modules_v2.py'deki etiket istatistigine gore valid/test'in
# ~%38-39'u kucuk nesne (<32px). Stride-4 dali tam bu dagilimi hedefliyor.
#
# AGIRLIK ZINCIRI - zayiftan iyiye, SON cagri kazanir.
#
#   kaynak     tek basina         benzersiz katkisi
#   baseline   355/902  %22.9     ortak iskelet
#   p2         894/902  %40.7     539 TENSOR: P2 kolu (19,20), bottom-up
#                                 (22,23,25,26,28) ve 4 SEVIYELI DETECT (29)
#   spdconv    360/902  %65.3     5 TENSOR ama parametrenin %42.4'u:
#                                 agir SPD backbone conv'lari (0,1,3,5,7)
#   ---------------------------------------------------------------------
#   zincir toplami       899/902 tensor, %83.1 parametre
#
# NEDEN p2 DE SART: baseline ve spdconv'un IKISININ DE Detect'i 3 seviyeli ve
# P2 kolu yok. O 539 tensoru baska hicbir kaynak dolduramaz; p2 zincirde
# olmazsa rastgele baslarlar.
#
# NEDEN spdconv SONDA: cakisan 355 tensor backbone BN'leri (0-10) ve top-down
# head (13,16). Modelin backbone'u SPD oldugu icin BN running istatistikleri
# SPD'nin urettigi aktivasyon dagilimindan gelmeli; p2'ninkiler duz Conv
# aktivasyonlarindan hesaplandi, uyumsuz.
# (spdconv'da katman 19 = P4/16-medium, bizde 19 = P2/4-xsmall gibi rol
#  uyusmazliklari var; ama gercek agirliklar shape tutmadigi icin zaten
#  atlaniyor, cakisan tek sey num_batches_tracked sayaclari.)
#
# BU ZINCIRIN OLCULEN ETKISI (eski baseline->spdconv zincirine gore):
#   1. epoch train/box_loss   3.17 -> 1.23   (P2 kolu + Detect rastgele degil)
#   valid mAP50             0.8885 -> 0.8898
#   valid mAP50-95          0.6427 -> 0.6457
#   test  mAP50              %88.65 -> %89.30
#   test  mAP50-95           %64.93 -> %65.39
CKPT = [
    ("baseline", "../baseline/runs/detect/train/weights/best.pt"),
    ("p2",       "../p2/runs/detect/ablation_p2_v/weights/best.pt"),
    ("spdconv",  "../SPD-Conv/runs/detect/ablation_spd_conv_v2/weights/best.pt"),
]

import os
from ultralytics.utils.torch_utils import intersect_dicts

model = YOLO("spd-covn-p2.yaml")
dst = model.model.state_dict()
total = sum(v.numel() for v in dst.values())
filled = set()

print(f"hedef: {len(dst)} tensor / {total/1e6:.2f}M parametre")
print()
print(f"{'kaynak':<10} {'eslesen':>8} {'YENI':>6} {'ezilen':>7} {'kumulatif param':>16}")
print("-" * 52)
for name, path in CKPT:
    if not os.path.exists(path):
        print(f"{name:<10} ATLANDI - dosya yok ({path})")
        continue
    ck = torch.load(path, map_location="cpu", weights_only=False)
    src = (ck.get("ema") or ck.get("model")).float().state_dict()
    inter = intersect_dicts(src, dst)
    new, over = set(inter) - filled, set(inter) & filled
    filled |= set(inter)
    model = model.load(path)
    print(f"{name:<10} {len(inter):>8} {len(new):>6} {len(over):>7} "
          f"{100*sum(dst[k].numel() for k in filled)/total:>15.1f}%")

print("-" * 52)
print(f"dolan: {len(filled)}/{len(dst)} tensor, rastgele kalan: {len(dst)-len(filled)}")
print("parametre: %.2fM" % (sum(p.numel() for p in model.model.parameters()) / 1e6))

In [ ]:
# --- SPD-Conv + P2 egitimi ---
# MALIYET: P2 dali stride-4 seviyesinde calisiyor, yani P3'un 4 kati
# aktivasyon alani. SPD-Conv da kanal sayisini gecici olarak 4x buyutuyor.
# Parametre 2.5M -> 4.55M. 8GB VRAM'de batch=16 sinirda olabilir;
# CUDA OOM alirsan batch=8 yap.
model.train(
    data=DATA,
    epochs=500,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    device=device,
    patience=100,
    plots=True,
    name="ablation_spd_conv_p2",
)

In [ ]:
import csv, glob
run = sorted(glob.glob("runs/detect/ablation_spd_conv_p2/results.csv"))[-1]
rows = list(csv.DictReader(open(run)))
print(run, "| epoch:", len(rows))
for r in rows[:2] + rows[-3:]:
    print("ep %-4s box=%-8s cls=%-8s mAP50=%-7s mAP50-95=%s" % (
        r["epoch"], r["train/box_loss"], r["train/cls_loss"],
        r["metrics/mAP50(B)"], r["metrics/mAP50-95(B)"]))

b = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
print("\nEN IYI epoch %s -> P %.3f R %.3f mAP50 %.4f mAP50-95 %.4f" % (
    b["epoch"], float(b["metrics/precision(B)"]), float(b["metrics/recall(B)"]),
    float(b["metrics/mAP50(B)"]), float(b["metrics/mAP50-95(B)"])))

box1 = float(rows[0]["train/box_loss"])
print("\n[saglik] 1. epoch train/box_loss = %.2f -> %s" % (
    box1, "OK" if box1 < 20 else "!! KAYIP OLCEGI PATLAMIS, DURDUR"))
if len(rows) < 500:
    print("[uyari] kosu %d epoch'ta bitmis, 500 degil - yarida kesilmis olabilir" % len(rows))

In [ ]:
from pathlib import Path
best = Path(sorted(glob.glob("runs/detect/ablation_spd_conv_p2/weights/best.pt"))[-1])
print("test ediliyor:", best)

res = YOLO(str(best)).val(
    data=DATA,
    split="test",
    imgsz=640,
    device=device,
)
d = res.results_dict
print("\n=== TEST ===")
for k, lbl in [("metrics/precision(B)", "Precision"),
               ("metrics/recall(B)", "Recall"),
               ("metrics/mAP50(B)", "mAP@0.5"),
               ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    v = d.get(k)
    print(f"{lbl:>14}: {v*100:.2f}%" if v is not None else f"{lbl:>14}: yok")

try:
    for i, c in enumerate(res.names.values()):
        print(f"  {c:>6}: mAP50={res.box.ap50[i]:.4f}  mAP50-95={res.box.ap[i]:.4f}")
except Exception as e:
    print("sinif bazinda metrik alinamadi:", e)